<a href="https://colab.research.google.com/github/Thonyta17/Midterm-Project-The-Replication-Study/blob/main/notebooks/02_Replication_Analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [254]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [255]:
#Load data from 01_Data_cleaning_ipynb

df = pd.read_csv('cleaned_data.csv')
df

,Store ID,Chain,Co-owned,State,South NJ,Central NJ,North NJ,Phil. suburb,Easton,NJ Shore,...,Raise increase ($/hr)_2,Special Program_2,Free/Reduced meals_2,Opening Hour_2,Operating hours_2,P_Soda_2,P_Fry_2,P_Entree_2,Num. Registers_2,Num. Registers at 11am_2
0,46,1,0,0,0,0,0,1,0,0,...,0.08,1,2,6.50,16.50,1.03,.,0.94,4,4
1,49,2,0,0,0,0,0,1,0,0,...,0.05,0,2,10.00,13.00,1.01,0.89,2.35,4,4
2,506,2,1,0,0,0,0,1,0,0,...,0.25,.,1,11.00,11.00,0.95,0.74,2.33,4,3
3,56,4,1,0,0,0,0,1,0,0,...,0.15,0,2,10.00,12.00,0.92,0.79,0.87,2,2
4,61,4,1,0,0,0,0,1,0,0,...,0.15,0,2,10.00,12.00,1.01,0.84,0.95,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
405,423,2,1,1,0,0,1,0,0,0,...,0.50,0,1,11.00,11.00,1.05,0.84,2.32,3,2
406,424,2,1,1,0,0,1,0,0,0,...,0.50,0,1,11.00,14.00,1.05,0.94,2.32,5,3
407,426,3,1,1,0,0,1,0,0,0,...,0.25,1,2,6.00,18.00,1.11,1.05,1.05,6,5
408,427,4,0,1,0,0,1,0,0,0,...,.,1,2,10.50,12.50,1.11,1.09,2.07,2,2


From the paper:

Table 2 presents the means for several key variables in our data set, averaged over the subset of nonmissing responses for each variable. In constructing the means, employment in wave 2 is set to 0 for the permanently closed stores but is treated as missing for the temporarily closed stores. (Fulltime-equivalent [FTE] employment was calculated as the number of full-time workers
[including managers] plus 0.5 times the number of part-time workers.)'


Second Interview Status


0 = refused second interview (count = 1)


1 = answered 2nd interview (count = 399)


2 = closed for renovations (count = 2)


3 = closed "permanently" (count = 6)


4 = closed for highway construction (count = 1)


5 = closed due to Mall fire (count = 1)

In [256]:
#covert columns to numeric
numeric_cols = [
    'FT Empl.', 'PT Empl.', 'Num. Mgrs',
    'FT Empl._2', 'PT Empl._2', 'Num. Mgrs_2',
    'Starting Wage', 'Starting Wage_2',
    'P_Soda', 'P_Soda_2',
    'P_Fry', 'P_Fry_2',
    'P_Entree', 'P_Entree_2',
    'Operating hours', 'Operating hours_2'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [257]:
#create FTE column according to Card & Krueger definition

# Permanently closed stores (Status 2nd Interview=3): set Wave 2 employment to 0
df.loc[df['Status 2nd Interview'] == 3, 'FT Empl._2']  = 0
df.loc[df['Status 2nd Interview'] == 3, 'PT Empl._2']  = 0
df.loc[df['Status 2nd Interview'] == 3, 'Num. Mgrs_2'] = 0

# Temporarily closed stores (Status 2nd Interview= 2, 4, 5): drop entirely
df = df[~df['Status 2nd Interview'].isin([2, 4, 5])].copy()

#create column
df['fte1'] = df['FT Empl.']   + 0.5 * df['PT Empl.']   + df['Num. Mgrs']
df['fte2'] = df['FT Empl._2'] + 0.5 * df['PT Empl._2'] + df['Num. Mgrs_2']

State: 1=NJ, 0= PA

In [258]:
# Create dummy NJ column to easier identify if state is NJ or not
df['nj']=df['State'].astype(int)


**1.Descriptive Table Reconstruction (Table 2)**

In [259]:
from scipy import stats

# define df for each state
nj = df[df.nj==1].copy()
pa = df[df.nj==0].copy()

In [260]:
#define calculation for mean and standard deviation
def mean_sd(s):
    s = s.dropna()
    if s.empty:
        return "N/A"
    return f"{s.mean():.2f} ({s.std():.2f})"

#calculate row in the table
rows = {
    'FTE Employment':  [mean_sd(nj['fte1']), mean_sd(pa['fte1']), mean_sd(nj['fte2']), mean_sd(pa['fte2'])],
    'Starting Wage':   [mean_sd(nj['Starting Wage']), mean_sd(pa['Starting Wage']), mean_sd(nj['Starting Wage_2']), mean_sd(pa['Starting Wage_2'])],
    'Price of Soda':   [mean_sd(nj['P_Soda']), mean_sd(pa['P_Soda']), mean_sd(nj['P_Soda_2']), mean_sd(pa['P_Soda_2'])],
    'Price of Fries':  [mean_sd(nj['P_Fry']), mean_sd(pa['P_Fry']), mean_sd(nj['P_Fry_2']), mean_sd(pa['P_Fry_2'])],
    'Price of Entree': [mean_sd(nj['P_Entree']), mean_sd(pa['P_Entree']), mean_sd(nj['P_Entree_2']), mean_sd(pa['P_Entree_2'])],
}


#create table
table2 = pd.DataFrame(rows,
    index=['NJ Before', 'PA Before', 'NJ After', 'PA After']
).T


print('=== Descriptive Table of key variable before and after wave ===\n')
print(table2.to_string())
print('***Values shown as: Mean (Std Dev)\n')

# Unit test
print(f"\nUnit Test — NJ FTE Before: {nj['fte1'].mean():.2f} (paper: 20.4)")
print(f"Unit Test — PA FTE Before: {pa['fte1'].mean():.2f} (paper: 23.3)")

=== Descriptive Table of key variable before and after wave ===

                    NJ Before      PA Before      NJ After      PA After
FTE Employment   20.47 (9.14)  23.33 (11.86)  21.03 (9.29)  21.17 (8.28)
Starting Wage     4.61 (0.35)    4.63 (0.35)   5.08 (0.10)   4.62 (0.36)
Price of Soda     1.06 (0.08)    0.97 (0.07)   1.06 (0.09)   0.98 (0.08)
Price of Fries    0.94 (0.10)    0.84 (0.09)   0.96 (0.10)   0.86 (0.10)
Price of Entree   1.35 (0.65)    1.22 (0.62)   1.40 (0.66)   1.19 (0.58)
***Values shown as: Mean (Std Dev)


Unit Test — NJ FTE Before: 20.47 (paper: 20.4)
Unit Test — PA FTE Before: 23.33 (paper: 23.3)


**2.The Simple Difference**

In [261]:
# Define variables for Wave 1 and Wave 2 column
variables = {
    'FTE Employment': ('fte1','fte2'),
    'Starting Wage':  ('Starting Wage','Starting Wage_2'),
    'Price of Soda':  ('P_Soda','P_Soda_2'),
    'Price of Fries': ('P_Fry','P_Fry_2'),
    'Price of Entree':('P_Entree','P_Entree_2'),
}

# Loop through each variable, compute means/std and DiD
rows = []
for var_name, (col1, col2) in variables.items():
    nj_before = nj[col1].dropna()
    nj_after  = nj[col2].dropna()
    pa_before = pa[col1].dropna()
    pa_after  = pa[col2].dropna()

    delta_nj = nj_after.mean() - nj_before.mean()
    delta_pa = pa_after.mean() - pa_before.mean()
    did      = delta_nj - delta_pa

    rows.append({
        'Variable':    var_name,
        'NJ Before':   f"{nj_before.mean():.2f} ({nj_before.std():.2f})",
        'NJ After':    f"{nj_after.mean():.2f} ({nj_after.std():.2f})",
        'NJ Change':   f"{delta_nj:+.2f}",
        'PA Before':   f"{pa_before.mean():.2f} ({pa_before.std():.2f})",
        'PA After':    f"{pa_after.mean():.2f} ({pa_after.std():.2f})",
        'PA Change':   f"{delta_pa:+.2f}",
        'DiD':         f"{did:+.2f}",
    })

result = pd.DataFrame(rows).set_index('Variable')

print('=== Simple Difference of Means ===')
print('    Values shown as: Mean (Std Dev)\n')
print(result.to_string())

=== Simple Difference of Means ===
    Values shown as: Mean (Std Dev)

                    NJ Before      NJ After NJ Change      PA Before      PA After PA Change    DiD
Variable                                                                                           
FTE Employment   20.47 (9.14)  21.03 (9.29)     +0.56  23.33 (11.86)  21.17 (8.28)     -2.17  +2.73
Starting Wage     4.61 (0.35)   5.08 (0.10)     +0.47    4.63 (0.35)   4.62 (0.36)     -0.01  +0.48
Price of Soda     1.06 (0.08)   1.06 (0.09)     +0.00    0.97 (0.07)   0.98 (0.08)     +0.00  -0.00
Price of Fries    0.94 (0.10)   0.96 (0.10)     +0.02    0.84 (0.09)   0.86 (0.10)     +0.02  +0.00
Price of Entree   1.35 (0.65)   1.40 (0.66)     +0.04    1.22 (0.62)   1.19 (0.58)     -0.03  +0.07


**3.Regression Implementation:**

In [262]:
import statsmodels.formula.api as smf

# Reshape wide to long format
# Each store appears twice: once for Wave 1 (post=0) and once for Wave 2 (post=1)
pre = df[['Store ID', 'nj', 'fte1']].rename(columns={'fte1': 'fte'})
pre['post'] = 0

post = df[['Store ID', 'nj', 'fte2']].rename(columns={'fte2': 'fte'})
post['post'] = 1

long = pd.concat([pre, post], ignore_index=True)
long

,Store ID,nj,fte,post
0,46,0,40.50,0
1,49,0,13.75,0
2,506,0,8.50,0
3,56,0,34.00,0
4,61,0,24.00,0
...,...,...,...,...
807,423,1,23.75,1
808,424,1,17.50,1
809,426,1,20.50,1
810,427,1,20.50,1


In [263]:
# Create Treat x Post
long['treat_post'] = long['nj'] * long['post']
long = long.dropna(subset=['fte']).reset_index(drop=True)

# Run regression
model = smf.ols('fte ~ nj + post + treat_post', data=long).fit()

print('=== DiD Regression Results ===\n')
print(model.summary().tables[1])
print(f'\nDiD estimate (treat_post): {model.params["treat_post"]:.2f}')
print(f'Paper benchmark:           +2.76')

=== DiD Regression Results ===

                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     23.3312      1.073     21.735      0.000      21.224      25.438
nj            -2.8657      1.197     -2.395      0.017      -5.215      -0.517
post          -2.1656      1.518     -1.427      0.154      -5.145       0.814
treat_post     2.7276      1.692      1.612      0.107      -0.594       6.049

DiD estimate (treat_post): 2.73
Paper benchmark:           +2.76


**4.Standard Errors and Clustering:**

In [265]:
#same regression as above but with cluster standard errors
long = pd.concat([pre, post], ignore_index=True)
long['treat_post'] = long['nj'] * long['post']
long = long.dropna(subset=['fte']).reset_index(drop=True)

model_clustered = smf.ols('fte ~ nj + post + treat_post', data=long).fit(
    cov_type='cluster',
    cov_kwds={'groups': long['Store ID']}
)

print('=== Step 4: DiD Regression with Clustered Standard Errors ===\n')
print(model_clustered.summary().tables[1])
print(f'\nDiD estimate (treat_post): {model_clustered.params["treat_post"]:.2f}')
print(f'Std. Error (clustered):    {model_clustered.bse["treat_post"]:.2f}')
print(f'Paper benchmark:           DiD ≈ 2.76 | SE ≈ 1.36')

=== Step 4: DiD Regression with Clustered Standard Errors ===

                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept     23.3312      1.347     17.326      0.000      20.692      25.970
nj            -2.8657      1.438     -1.992      0.046      -5.685      -0.047
post          -2.1656      1.218     -1.778      0.075      -4.553       0.222
treat_post     2.7276      1.307      2.087      0.037       0.166       5.289

DiD estimate (treat_post): 2.73
Std. Error (clustered):    1.31
Paper benchmark:           DiD ≈ 2.76 | SE ≈ 1.36
